# Week 10 — Bayesian Optimization

Generate optimized recommendations for Week 10 using utility modules.

## Setup

In [ ]:
import numpy as np
import warnings
import sys
import importlib
sys.path.append('..')  # Add parent directory to path

import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import propose_next_point, fit_gp, get_strategy

from utils.data_utils import (
    load_week_data,
    save_week_data,
    combine_with_week_results, 
    print_data_summary
)

## 1. Load Week 9 Data

In [ ]:
inputs, outputs = load_week_data("../week 9/week9_clean_data.npz")
print_data_summary(inputs, outputs, "Week 9 Data")

## 2. Add Week 9 Results

In [ ]:
# Week 9 submitted points (actual values submitted)
week9_inputs = {
    1: np.array([0.428000, 0.420000]),
    2: np.array([0.200000, 0.500000]),
    3: np.array([0.347863, 0.672420, 0.435172]),
    4: np.array([0.443690, 0.393340, 0.368202, 0.443441]),
    5: np.array([0.000000, 1.000000, 1.000000, 1.000000]),
    6: np.array([0.755469, 0.276580, 0.644099, 0.672228, 0.162862]),
    7: np.array([0.000000, 0.312263, 0.707844, 0.246481, 0.405689, 0.758028]),
    8: np.array([0.107475, 0.120302, 0.040000, 0.207071, 1.000000, 0.099881, 0.180000, 0.998620])
}

# Week 9 outputs (received from black box)
week9_outputs = {
    1: 0.8974824842036074,
    2: -0.0014945301211180876,
    3: -0.007509698033295065,
    4: -0.04318375962393217,
    5: 4440.5225,
    6: -0.5672071068089778,
    7: 1.8696127261716888,
    8: 9.774664173904
}

# Combine with Week 9 results
inputs, outputs = combine_with_week_results(inputs, outputs, week9_inputs, week9_outputs)
print_data_summary(inputs, outputs, "After Week 9 Results")

In [ ]:
save_week_data(inputs, outputs, "week10_clean_data.npz")

## 3. Week 9 Results Analysis

In [ ]:
print("=" * 70)
print("WEEK 9 RESULTS ANALYSIS")
print("=" * 70)

best_before_w9 = {}
for fid in range(1, 9):
    best_before_w9[fid] = np.max(outputs[fid][:-1])

print(f"\n{'F':>2} {'Dims':>4} {'Best Before W9':>14} {'W9 Query':>14} {'New Best':>14} {'Status'}")
print("-" * 70)

improved = 0
for fid in range(1, 9):
    dim = inputs[fid].shape[1]
    prev_best = best_before_w9[fid]
    w9_val = week9_outputs[fid]
    new_best = np.max(outputs[fid])
    
    if w9_val >= prev_best:
        status = "NEW BEST"
        improved += 1
    else:
        status = f"miss (best still {prev_best:.4f})"
    
    print(f"{fid:>2} {dim:>3}D {prev_best:>14.4f} {w9_val:>14.4f} {new_best:>14.4f}   {status}")

print(f"\nWeek 9 hit rate: {improved}/8 functions improved")
print("=" * 70)

In [ ]:
# Detailed Week 9 strategy evaluation
print("=" * 70)
print("WEEK 9 STRATEGY EVALUATION")
print("=" * 70)

strategies_w9 = {
    1: ("COMBO both dims +0.005", "MISS 0.897 vs best 0.899. Combo overshot — narrow spike can't handle simultaneous changes. Revert to single-dim."),
    2: ("DEEP EXPLORE [0.20, 0.50]", "MISS -0.001. dim1=0.20 is a dead zone. Second peak not in low-dim1 region. Try mid-range dim1=0.45 next."),
    3: ("FRESH DIM dim3-0.004", "MISS -0.0075 vs best -0.0056. dim3 decrease is wrong direction. Try dim3+0.004 next."),
    4: ("GP/EI tight bounds ±0.03", "DISASTER -0.043 vs best 0.724. GP predicted 1.086 — completely wrong. NEVER trust EI on F4 again."),
    5: ("FACTORIAL [0,1,1,1]", "INFO: 4441 = same as [1,0,1,1]. Dims 1&2 are interchangeable, each contributes ~4220."),
    6: ("PRECISION dim2+0.001", "MISS -0.567 vs best -0.521. dim2=0.277 worse. Optimum is at or below 0.276."),
    7: ("PROVEN dim2-0.002", "NEW BEST +0.003 (1.866→1.870). 4th consecutive win! Most reliable strategy in the project."),
    8: ("BOLD dim3+0.020", "NEW BEST +0.012 (9.763→9.775). Fresh dimension paid off! Biggest F8 gain since W3→W4.")
}

for fid in range(1, 9):
    strategy, evaluation = strategies_w9[fid]
    print(f"\nF{fid}: {strategy}")
    print(f"  → {evaluation}")

print(f"\n{'=' * 70}")
print("SUMMARY: 2/8 new bests (F7, F8)")
print("Lessons: F1 combo failed (revert to single-dim), F4 EI catastrophic (never again), F8 dim3 works")
print("=" * 70)

In [ ]:
# Cumulative best scores
print("=" * 70)
print("CUMULATIVE BEST SCORES ACROSS ALL WEEKS")
print("=" * 70)

print(f"\n{'F':>2} {'W1':>8} {'W2':>8} {'W3':>8} {'W4':>8} {'W5':>8} {'W6':>8} {'W7':>8} {'W8':>8} {'W9':>8}")
print("-" * 85)

week_results = {
    1: [0.0979, 3.1e-39, 0.3255, 0.4147, 0.6114, 0.7062, 0.8787, 0.8992, 0.8975],
    2: [0.5567, 0.6138, 0.0480, 0.6050, 0.5471, 0.5725, 0.5824, 0.1273, -0.0015],
    3: [-0.0593, -0.0499, -0.1750, -0.0056, -0.0701, -0.0079, -0.0077, -0.0072, -0.0075],
    4: [-4.4163, 0.3523, 0.4226, 0.6723, 0.7101, 0.4657, 0.7243, 0.7152, -0.0432],
    5: [1231.61, 1688.07, 7599.50, 8662.48, 8290.38, 8643.15, 1616.64, 4440.52, 4440.52],
    6: [-0.5920, -0.5210, -1.0560, -0.5902, -0.9899, -0.5207, -0.6062, -0.5301, -0.5672],
    7: [1.3646, 1.7845, 1.3720, 1.7718, 1.4720, 1.8536, 1.8617, 1.8665, 1.8696],
    8: [9.5863, 9.6493, 9.6972, 9.7627, 9.7274, 9.7402, 9.7614, 9.7621, 9.7747]
}

for fid in range(1, 9):
    running_best = []
    current_best = float('-inf')
    for val in week_results[fid]:
        current_best = max(current_best, val)
        running_best.append(current_best)
    vals = ' '.join(f'{v:>8.4f}' for v in running_best)
    print(f"{fid:>2} {vals}")

print(f"\n{'F':>2} {'Overall Best':>14} {'Best Week':>10}")
print("-" * 30)
for fid in range(1, 9):
    best_val = max(week_results[fid])
    best_week = week_results[fid].index(best_val) + 1
    print(f"{fid:>2} {best_val:>14.4f} {'W' + str(best_week):>10}")
print("=" * 70)

## 4. Sensitivity Analysis

In [ ]:
from utils.sensitivity import sensitivity_analysis

for func_id in range(1, 9):
    sensitivity_analysis(func_id, inputs[func_id], outputs[func_id])

## 5. Week 10 Strategy Design — EXPERIMENTAL, NO WASTED QUERIES

### Philosophy shift (peer-informed):
- **Nikolas**: "A landscape that looked unimodal can suddenly support multimodality once points land in empty regions" — our exploration queries could reshape GP beliefs entirely
- **Santosh**: F5 showed "vertical jump" after exploration — wild probes can trigger breakthroughs
- **Bilal**: Portfolio approach — anchor queries (F1, F7) + bold probes (F2, F4, F5, F6)
- **Us**: We've been playing 7/10 for weeks. Time to chase 10/10 on stuck functions.

### Strategy per function:

| F | Strategy | Rationale | Risk/Reward |
|---|----------|-----------|-------------|
| F1 | **dim2+0.010 → [0.423, 0.425]** | Double step. Gains shrinking at +0.005. Test if bigger step catches more of the spike. | MED risk, HIGH reward |
| F2 | **[0.45, 0.50]** — mid-gap exploration | dim1=0.20 dead, 0.68-0.81 exploited. Midpoint of unexplored range. Could trigger GP multimodality (Nikolas). | HIGH risk, HIGH reward |
| F3 | **dim3+0.004 → [0.348, 0.672, 0.443]** | dim3 decrease failed W9. Reverse direction. First test of dim3+. | MODERATE |
| F4 | **dim1-0.005 AND dim4-0.005** | Smooth landscape (ls~1.5). Two fresh dims simultaneously. EI banned. Like F8's dim3 breakthrough but double. | MED risk, HIGH reward |
| F5 | **[0.5, 0.5, 0.5, 0.5]** — dead centre | Completely unexplored region. Could reveal hidden structure (Santosh's vertical jump). Wild card. | HIGH risk, HIGH reward |
| F6 | **dim4+0.0002** — ultra-sensitive breakthrough | dim2 exhausted (4 weeks, ~0.01 variation). dim4 ls=0.001, so 0.0002 = 1/5 length scale. High ceiling. | HIGH risk, HIGH reward |
| F7 | **dim2-0.003 → [0.0, 0.309, ...]** | Accelerate winning streak. 4 bests at -0.005, -0.005, -0.003, -0.002. Try -0.003. Still above 0.271. | LOW risk, HIGH confidence |
| F8 | **dim3+0.030 → [0.107, 0.120, 0.070, ...]** | W9 breakthrough. Accelerate to close 0.18 peer gap. Bolder step. | MED risk, HIGH reward |

In [ ]:
import warnings
import importlib
import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import fit_gp, propose_next_point

week10_recommendations = {}

def get_best_point(fid):
    """Get the best observed point for a function"""
    best_idx = np.argmax(outputs[fid])
    return inputs[fid][best_idx].copy()

# ============================================================
# F1: BIGGER STEP — dim2+0.010
# +0.005 gains shrinking (W7:+0.173, W8:+0.021, W9 combo failed).
# Double the step to test if we can jump further up the spike.
# W8 best: [0.423, 0.415] → 0.899. dim2+0.010 → 0.425.
# ============================================================
best1 = get_best_point(1)  # [0.423, 0.415] → 0.8992
week10_recommendations[1] = np.array([best1[0], best1[1] + 0.010])

# ============================================================
# F2: MID-GAP EXPLORATION — [0.45, 0.50]
# dim1=0.20 dead, 0.68-0.81 exploited. Gap 0.20-0.68 untested.
# Midpoint 0.45. Could trigger GP multimodality (Nikolas insight).
# dim2 irrelevant (ls=1.015).
# ============================================================
week10_recommendations[2] = np.array([0.450000, 0.500000])

# ============================================================
# F3: REVERSE — dim3+0.004
# dim3-0.004 failed W9 (-0.0075). Try opposite direction.
# Best dim3=0.439. +0.004 → 0.443. Full length scale step.
# ============================================================
best3 = get_best_point(3)  # [0.347863, 0.672420, 0.439172] → -0.0056
week10_recommendations[3] = np.array([best3[0], best3[1], best3[2] + 0.004])

# ============================================================
# F4: TWO FRESH DIMS — dim1-0.005 AND dim4-0.005
# EI banned (catastrophe twice). Smooth landscape (all ls~1.5).
# Both dims untested from best. Multi-dim safe on smooth functions.
# Like F8's dim3 breakthrough — fresh dimensions unlock gains.
# ============================================================
best4 = get_best_point(4)  # [0.413690, 0.367443, 0.360391, 0.413441] → 0.7243
week10_recommendations[4] = np.array([
    best4[0] - 0.005,    # dim1 -0.005 (0.414→0.409) — FRESH
    best4[1],            # dim2 locked
    best4[2],            # dim3 locked at sweet spot 0.360
    best4[3] - 0.005     # dim4 -0.005 (0.413→0.408) — FRESH
])

# ============================================================
# F5: DEAD CENTRE — [0.5, 0.5, 0.5, 0.5]
# Completely unexplored region. All prior queries at boundaries.
# Santosh's F5 showed "vertical jump" after exploration.
# Could reveal hidden structure in the middle of the space.
# ============================================================
week10_recommendations[5] = np.array([0.500000, 0.500000, 0.500000, 0.500000])

# ============================================================
# F6: ULTRA-SENSITIVE — dim4+0.0002
# dim2 exhausted (0.275/0.276/0.277 all tested, ~0.05 variation).
# dim4 has ls=0.001. 0.0002 = 1/5 of length scale.
# Unlike W5's disaster (ALL dims moved), this is single-dim.
# High ceiling — this is where untapped gains live.
# ============================================================
best6 = get_best_point(6)  # [0.755469, 0.275580, 0.644099, 0.672228, 0.162862] → -0.5207
week10_recommendations[6] = np.array([
    best6[0],            # dim1 locked
    best6[1],            # dim2 locked (exhausted)
    best6[2],            # dim3 locked
    best6[3] + 0.0002,   # dim4 +0.0002 (1/5 of ls=0.001) — BOLD
    best6[4]             # dim5 locked
])

# ============================================================
# F7: ACCELERATE — dim2-0.003
# 4 consecutive bests. Steps were -0.005, -0.005, -0.003, -0.002.
# Accelerate back to -0.003. 0.312→0.309. Still above 0.271 danger.
# ============================================================
best7 = get_best_point(7)  # [0.0, 0.312263, ...] → 1.8696
week10_recommendations[7] = np.array([
    best7[0],
    best7[1] - 0.003,    # dim2 -0.003 (0.312→0.309) — ACCELERATE
    best7[2],
    best7[3],
    best7[4],
    best7[5]
])

# ============================================================
# F8: BOLD CONTINUE — dim3+0.030
# W9's +0.020 gave new best (+0.012). Accelerate.
# dim3: 0.020→0.040→now 0.070. Bolder step to close peer gap.
# Peer at 9.96, we're at 9.775. Need bigger moves.
# ============================================================
best8 = get_best_point(8)  # W9 best: dim3=0.040
week10_recommendations[8] = np.array([
    best8[0],
    best8[1],
    best8[2] + 0.030,    # dim3 +0.030 (0.040→0.070) — BOLD
    best8[3],
    best8[4],
    best8[5],
    best8[6],
    best8[7]
])

# ============================================================
# Sanity check
# ============================================================
print("Week 10 Recommendations — EXPERIMENTAL")
print("=" * 80)
for fid in range(1, 9):
    X, y = inputs[fid], outputs[fid]
    rec = week10_recommendations[fid]
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        gp = fit_gp(X, y)
    pred, pred_std = gp.predict(rec.reshape(1, -1), return_std=True)
    dists = np.linalg.norm(X - rec, axis=1)
    min_dist = np.min(dists)
    best = np.max(y)
    
    strategy = {
        1: "BIGGER STEP dim2+0.010 — test step limit",
        2: "MID-GAP [0.45, 0.50] — trigger multimodality",
        3: "REVERSE dim3+0.004 — opposite of W9 failure",
        4: "TWO FRESH DIMS dim1&dim4 -0.005 — smooth landscape",
        5: "DEAD CENTRE [0.5,0.5,0.5,0.5] — unexplored wild card",
        6: "ULTRA-SENSITIVE dim4+0.0002 — breakthrough potential",
        7: "ACCELERATE dim2-0.003 → 0.309 — 5th best target",
        8: "BOLD dim3+0.030 → 0.070 — close peer gap"
    }
    
    print(f"F{fid} ({X.shape[1]}D)  best={best:.4f}  pred={pred[0]:.4f}±{pred_std[0]:.4f}  dist={min_dist:.4f}  {strategy[fid]}")
    print(f"  point: {rec}")
print("=" * 80)

## 6. Submission Format

In [ ]:
print("=" * 70)
print("WEEK 10 SUBMISSION")
print("=" * 70)

for fid in range(1, 9):
    pt = week10_recommendations[fid]
    formatted = '-'.join(f'{x:.6f}' for x in pt)
    print(f"Function {fid}:\t{formatted}")